In [2]:
from training_utilities_2nd_part import *

In [3]:
#oil
from variables_to_specify_oil import *

df, columns_to_normalize, oil_target_col, forecast_avg_target_col_name, avg_target_col_name, No_of_datapoints_in_one_day, start_date, end_date, delta, one_month_days, out_columns, oil_drop_columnss, oil_windows, index_of_one_month, one_month_window_size, date_col_name = variables_to_specify_oil()


from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])

df = df.dropna().reset_index(drop=True)

convert_time(df, date_col_name)

oil_df = df
oil_time_steps = 1

# stationary

In [5]:
oil_len_of_training_data_of_stationary_model =7*No_of_datapoints_in_one_day

train = df[0:oil_len_of_training_data_of_stationary_model] 
test = df[oil_len_of_training_data_of_stationary_model:]

eval_df_first_month, stationary_model1 = stationary_model_with_hptuning(train, test, one_month_window_size, 1, out_columns, oil_target_col, oil_drop_columnss)

sum_training_time_stat1 = eval_df_first_month['training_time'].sum()
print('sum_training_time is: ', sum_training_time_stat1)

print(eval_df_first_month['Testing Error'].mean())

Model Type: RandomForestRegressor
Storage Required: 1.45 MB
model storage is : 1.4451608657836914


total_time is:  0.2148309579999994
sum_training_time is:  1.6433725000000106
0.12797255040117117


# Model reuse

In [6]:
# Model reuse
daily_df_avg = get_elect_daily_avg(oil_df, No_of_datapoints_in_one_day, oil_target_col, avg_target_col_name)


seasonality_periods_acf_ls, seasonality_periods_acf, segmented_daily_df_avg, filtered_most_similar_dict_wass, filtered_most_similar_dict_tvd, forecast_daily_df_avg, segmented_forecast_daily_df_avg, filtered_forecasted_most_similar_dict_wass, filtered_forecasted_most_similar_dict_tvd = get_seasonality_segments_and_similarities(daily_df_avg, avg_target_col_name, forecast_avg_target_col_name, 7)

Detected seasonality periods (ACF): [  7 297 309 314 317 327 330 341 345 351 359]
median_value is:  327


## drift detection

In [7]:
df_copy = oil_df[[oil_target_col]]
target_col = oil_target_col
time_steps = oil_time_steps

df_copy['date'] = pd.to_datetime(df_copy.index)
multiplier = No_of_datapoints_in_one_day
x = 7* multiplier
window_len_=[x]
drift_results_df_ls = []
for i in window_len_:
    start_drift_detection_time = timeit.default_timer()
    drift_results_df = detect_drift_univariate(
        df_copy,
        target_col=oil_target_col,
        window_lengths=window_len_,
        arima_order=(1, 0, 0)
    )
    drift_results_df_ls.append(drift_results_df)
    drift_detection_time = timeit.default_timer() - start_drift_detection_time
    num_true = drift_results_df['drift_detected'].sum()
    print("i is: ", i, " and the Number of True values in 'drift_detected':", num_true, " total number of rows are : ", len(drift_results_df))
    print("drift detection time is: ", drift_detection_time)
    drift_results_df = drift_results_df_ls[0]
    drift_indices = list(drift_results_df.index[drift_results_df['drift_detected']])
    print("indices are: ", drift_indices)

Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Train size=33, Test size=33
Fold 1: Train size=66, Test size=33
Fold 2: Train size=99, Test size=33
Fold 3: Train size=132, Test size=33
Skipping fold 4: Insufficient training or test data.
Fold 0: Tr

In [9]:
eval_df_monthly2, avg_ml_storage1 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_wass, stationary_model1, oil_len_of_training_data_of_stationary_model,oil_df, "SA", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)


window is:  336
i/window is :  1.0
Model Type: RandomForestRegressor
Storage Required: 2.84 MB


window is:  672
i/window is :  2.0
Model Type: RandomForestRegressor
Storage Required: 2.86 MB


window is:  1008
i/window is :  3.0
similar_month_index is :  0
month_index:  3




window is:  1344
i/window is :  4.0
similar_month_index is :  0
month_index:  4




window is:  1680
i/window is :  5.0
Model Type: RandomForestRegressor
Storage Required: 2.64 MB


window is:  2016
i/window is :  6.0
Model Type: RandomForestRegressor
Storage Required: 2.82 MB


window is:  2352
i/window is :  7.0
Model Type: RandomForestRegressor
Storage Required: 2.81 MB


window is:  2688
i/window is :  8.0
Model Type: RandomForestRegressor
Storage Required: 2.78 MB


window is:  3024
i/window is :  9.0
Model Type: RandomForestRegressor
Storage Required: 2.64 MB


window is:  3360
i/window is :  10.0
similar_month_index is :  8
month_index:  10




window is:  3696
i/window is :  11.0
Model Type: RandomForestR

In [10]:
eval_df_monthly2, avg_ml_storage2 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_most_similar_dict_tvd, stationary_model1, oil_len_of_training_data_of_stationary_model,oil_df, "SA", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)

window is:  336
i/window is :  1.0
Model Type: RandomForestRegressor
Storage Required: 2.84 MB


window is:  672
i/window is :  2.0
Model Type: RandomForestRegressor
Storage Required: 2.86 MB


window is:  1008
i/window is :  3.0
Model Type: RandomForestRegressor
Storage Required: 2.63 MB


window is:  1344
i/window is :  4.0
Model Type: RandomForestRegressor
Storage Required: 2.82 MB


window is:  1680
i/window is :  5.0
similar_month_index is :  2
month_index:  5




window is:  2016
i/window is :  6.0
Model Type: RandomForestRegressor
Storage Required: 2.82 MB


window is:  2352
i/window is :  7.0
similar_month_index is :  5
previous_model_i is :  2352
math.floor(previous_model_i/window) is:  7
len(models_ls) is: 6
Model Type: RandomForestRegressor
Storage Required: 2.78 MB


window is:  2688
i/window is :  8.0
similar_month_index is :  2
month_index:  8




window is:  3024
i/window is :  9.0
similar_month_index is :  2
month_index:  9




window is:  3360
i/window is :  10.0
simil

In [11]:
eval_df_monthly2, avg_ml_storage3 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_wass, stationary_model1, oil_len_of_training_data_of_stationary_model,oil_df, "ES", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)

window is:  336
Model Type: RandomForestRegressor
Storage Required: 2.84 MB


window is:  672
Model Type: RandomForestRegressor
Storage Required: 2.86 MB


window is:  1008
Model Type: RandomForestRegressor
Storage Required: 2.63 MB


window is:  1344
similar_month_index is :  0
month_index:  3




window is:  1680
Model Type: RandomForestRegressor
Storage Required: 2.64 MB


window is:  2016
Model Type: RandomForestRegressor
Storage Required: 2.82 MB


window is:  2352
Model Type: RandomForestRegressor
Storage Required: 2.81 MB


window is:  2688
Model Type: RandomForestRegressor
Storage Required: 2.78 MB


window is:  3024
Model Type: RandomForestRegressor
Storage Required: 2.64 MB


window is:  3360
Model Type: RandomForestRegressor
Storage Required: 2.83 MB


window is:  3696
similar_month_index is :  8
month_index:  10




window is:  4032
Model Type: RandomForestRegressor
Storage Required: 2.37 MB


window is:  4368
similar_month_index is :  10
previous_model_i is :  4368
math.fl

In [12]:
eval_df_monthly2, avg_ml_storage4 = new_copied_reuse_with_hptuning_no_while_loop_with_drift(filtered_forecasted_most_similar_dict_tvd, stationary_model1, oil_len_of_training_data_of_stationary_model,oil_df, "ES", oil_target_col, oil_drop_columnss, oil_time_steps, seasonality_periods_acf, No_of_datapoints_in_one_day, drift_indices, 1)

window is:  336
Model Type: RandomForestRegressor
Storage Required: 2.84 MB


window is:  672
Model Type: RandomForestRegressor
Storage Required: 2.86 MB


window is:  1008
Model Type: RandomForestRegressor
Storage Required: 2.63 MB


window is:  1344
Model Type: RandomForestRegressor
Storage Required: 2.82 MB


window is:  1680
Model Type: RandomForestRegressor
Storage Required: 2.64 MB


window is:  2016
Model Type: RandomForestRegressor
Storage Required: 2.82 MB


window is:  2352
Model Type: RandomForestRegressor
Storage Required: 2.81 MB


window is:  2688
Model Type: RandomForestRegressor
Storage Required: 2.78 MB


window is:  3024
Model Type: RandomForestRegressor
Storage Required: 2.64 MB


window is:  3360
Model Type: RandomForestRegressor
Storage Required: 2.83 MB


window is:  3696
Model Type: RandomForestRegressor
Storage Required: 2.86 MB


window is:  4032
Model Type: RandomForestRegressor
Storage Required: 2.37 MB


window is:  4368
Model Type: RandomForestRegressor
Sto

In [16]:
avg_ml_storage_reuse = (avg_ml_storage1+avg_ml_storage2+avg_ml_storage3+avg_ml_storage4)/4
print(avg_ml_storage_reuse)

2.742014117311374


# informed

In [14]:
lstm_informed_update(stationary_model1,oil_df, target_col, oil_drop_columnss,time_steps, seasonality_periods_acf,No_of_datapoints_in_one_day, drift_indices, 1)

window is:  336
Model Type: RandomForestRegressor
Storage Required: 2.84 MB
window is:  672
Model Type: RandomForestRegressor
Storage Required: 2.86 MB
window is:  1008
Model Type: RandomForestRegressor
Storage Required: 2.63 MB
window is:  1344
window is:  1680
window is:  2016
window is:  2352
window is:  2688
window is:  3024
window is:  3360
window is:  3696
Model Type: RandomForestRegressor
Storage Required: 2.86 MB
window is:  4032
Model Type: RandomForestRegressor
Storage Required: 2.37 MB
window is:  4368
Model Type: RandomForestRegressor
Storage Required: 2.84 MB
window is:  4704
window is:  5040
Model Type: RandomForestRegressor
Storage Required: 2.81 MB
window is:  5376
window is:  5712
Model Type: RandomForestRegressor
Storage Required: 2.85 MB
window is:  6048
window is:  6384
Model Type: RandomForestRegressor
Storage Required: 2.83 MB
window is:  6720
Model Type: RandomForestRegressor
Storage Required: 2.66 MB
window is:  7056
window is:  7392
window is:  7728
window is: 

# periodical

In [15]:
periodical_retraining_with_hptuning(1, oil_df, oil_windows, out_columns, oil_target_col, oil_drop_columnss)

window size is :  120
Model Type: RandomForestRegressor
Storage Required: 1.02 MB
Model Type: RandomForestRegressor
Storage Required: 1.03 MB
Model Type: RandomForestRegressor
Storage Required: 1.00 MB
Model Type: RandomForestRegressor
Storage Required: 1.03 MB
Model Type: RandomForestRegressor
Storage Required: 1.01 MB
Model Type: RandomForestRegressor
Storage Required: 1.03 MB
Model Type: RandomForestRegressor
Storage Required: 0.84 MB
Model Type: RandomForestRegressor
Storage Required: 1.00 MB
Model Type: RandomForestRegressor
Storage Required: 1.03 MB
Model Type: RandomForestRegressor
Storage Required: 1.00 MB
Model Type: RandomForestRegressor
Storage Required: 1.01 MB
Model Type: RandomForestRegressor
Storage Required: 1.02 MB
Model Type: RandomForestRegressor
Storage Required: 0.83 MB
Model Type: RandomForestRegressor
Storage Required: 0.99 MB
Model Type: RandomForestRegressor
Storage Required: 0.99 MB
Model Type: RandomForestRegressor
Storage Required: 1.00 MB
Model Type: Random

([          Training dataset     Testing dataset       mae       mse      rmse  \
  0    trained on window i-1  tested on window i  0.031888  0.002076  0.045564   
  1    trained on window i-1  tested on window i  0.094025  0.011955  0.109340   
  2    trained on window i-1  tested on window i  0.203070  0.046050  0.214594   
  3    trained on window i-1  tested on window i  0.072248  0.007402  0.086035   
  4    trained on window i-1  tested on window i  0.035847  0.002100  0.045822   
  ..                     ...                 ...       ...       ...       ...   
  139  trained on window i-1  tested on window i  0.019213  0.000529  0.023009   
  140  trained on window i-1  tested on window i  0.029111  0.001452  0.038100   
  141  trained on window i-1  tested on window i  0.044449  0.002506  0.050055   
  142  trained on window i-1  tested on window i  0.039544  0.002545  0.050448   
  143  trained on window i-1  tested on window i  0.030292  0.001562  0.039516   
  
             